# Lab 1 — Probability, Odds & Log-Odds (Logit)

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Learning objectives

1. Compute **probability** from observed default rates in Lending Club data.
2. Convert probability to **odds** and **log-odds (logit)**.
3. Apply the **inverse logit** to recover probability.
4. Connect these quantities to **logistic regression** (Lab 2).

> **Checkpoints:** **1,000** rows · P(default) ≈ **0.49** · odds ≈ **0.94** · logit ≈ **-0.06**

<!-- cisco-day03-lab01-expanded-2026 -->

**Dataset:** `data/lending-club/lending_club_sample.csv` (**1,000** rows)

**Day 3 flow:** **Lab 1 (you are here) — Probability** → [Lab 2 — Logistic regression](lab02_logistic_regression.ipynb) → [Lab 3 — Confusion matrix](lab03_confusion_matrix.ipynb) → [Lab 4 — ROC & AUC](lab04_roc_auc.ipynb) → [Lab 5 — sklearn Pipeline](lab05_sklearn_pipeline.ipynb) → [Lab 6 — SHAP](lab06_shap_interpretability.ipynb)

## From regression to classification

| Day 2 (Zomato) | Day 3 (Lending Club) |
|----------------|----------------------|
| Target: continuous `aggregate_rating` | Target: binary `default` |
| Model: Linear Regression | Model: Logistic Regression (Lab 2) |
| Metric: RMSE, R² | Metric: precision, recall, AUC (Labs 3–4) |

Today we build the **probability language** logistic regression uses internally.

### Key definitions

| Term | Symbol | Formula | Range |
|------|--------|---------|-------|
| Probability | $p$ | successes / total | 0 to 1 |
| Odds | — | $p / (1-p)$ | 0 to ∞ |
| Log-odds (logit) | — | $\ln(p / (1-p))$ | −∞ to +∞ |

**Intuition:** Odds = 1 means 50/50. Odds = 2 means 2:1 in favor. Logistic regression models log-odds as a **linear** function of features.

## Why probability matters for lending

A credit analyst asks: *What fraction of loans like this one go bad?* That fraction is an **empirical probability**. Pricing (`int_rate`), underwriting rules, and model scores all start from the same arithmetic — count defaults, divide by total.

---

## 0. Locate the Lending Club file

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

LENDING_CLUB_CSV = GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
CATEGORICAL_FEATURES = ["grade", "term"]

print("GH root:", GH_ROOT)
print("CSV exists:", LENDING_CLUB_CSV.is_file())
print("Path:", LENDING_CLUB_CSV.relative_to(GH_ROOT))


### 0b. Peek at columns before loading

In [ ]:
peek = pd.read_csv(LENDING_CLUB_CSV, nrows=5)
print("Columns:", list(peek.columns))
display(peek)


---

## 1. Load data and define `default`

In [ ]:
df = pd.read_csv(LENDING_CLUB_CSV)
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

print(f"rows: {len(df)}")
print(df["loan_status"].value_counts())


We label **`default = 1`** for distressed loans (`Charged Off` or `Late (31-120 days)`). All other statuses are **0** (paid or current).

This is a simplification of the full Kaggle Lending Club schema — appropriate for classroom classification.

### 1b. Which statuses map to default?

In [ ]:
default_rows = df[df["default"] == 1]
print("Statuses counted as default:")
print(default_rows["loan_status"].value_counts())
print("\nStatuses counted as non-default (sample):")
print(df[df["default"] == 0]["loan_status"].value_counts().head(5))


### 1c. Numeric feature snapshot

In [ ]:
display(df[["loan_amnt", "int_rate", "annual_inc", "dti", "installment", "default"]].describe().round(2))


---

## 2. Empirical probability of default

In [ ]:
p_default = df["default"].mean()
p_paid = 1 - p_default

print(f"P(default): {p_default:.4f}")
print(f"P(fully current/paid): {p_paid:.4f}")

assert len(df) == 1000
print("✓ Row count checkpoint OK")


### 2b. Count defaults manually

In [ ]:
n_default = df["default"].sum()
n_safe = len(df) - n_default
print(f"defaults: {n_default}, non-defaults: {n_safe}")
print(f"manual P(default): {n_default / len(df):.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
sns.countplot(data=df, x="default", ax=ax, palette=["steelblue", "salmon"])
ax.set_xticks([0, 1])
ax.set_xticklabels(["No default (0)", "Default (1)"])
ax.set_title("Class balance — nearly 50/50 in lab sample")
plt.tight_layout()
plt.show()


Unlike Day 6 credit-card fraud (**99:1** imbalance), our Lending Club lab sample is **roughly balanced** — accuracy is more meaningful here.

### 2c. Default rate by loan grade

In [ ]:
grade_rate = df.groupby("grade")["default"].mean().sort_index()
print("P(default) by grade:")
print(grade_rate.round(3))


### 2d. Default rate by term

In [ ]:
term_rate = df.groupby("term")["default"].mean()
print(term_rate.round(3))


### 2e. Interest rate vs default (preview of Lab 2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.boxplot(data=df, x="default", y="int_rate", ax=ax, palette="Set2")
ax.set_xticklabels(["No default", "Default"])
ax.set_title("Higher int_rate tends to accompany defaults")
plt.tight_layout()
plt.show()


---

## 3. Odds of default

$$
\text{odds} = \frac{p}{1-p}
$$

In [ ]:
odds_default = p_default / p_paid
print(f"odds of default: {odds_default:.4f}")

manual_odds = df["default"].sum() / (len(df) - df["default"].sum())
print(f"manual odds (defaults / non-defaults): {manual_odds:.4f}")
assert abs(odds_default - manual_odds) < 0.001


**Read the number:** odds ≈ **0.94** means slightly **fewer** defaults than non-defaults (odds < 1).

### 3b. Odds when P = 0.50

In [ ]:
p_half = 0.50
odds_half = p_half / (1 - p_half)
print(f"P=0.50 → odds = {odds_half:.2f} (even money)")


### 3c. Portfolio odds vs grade E odds

In [ ]:
e_grade = df[df["grade"] == "E"]
if len(e_grade) > 0:
    p_e = e_grade["default"].mean()
    odds_e = p_e / (1 - p_e)
    print(f"Grade E: P(default)={p_e:.3f}, odds={odds_e:.3f}")
print(f"Full portfolio odds: {odds_default:.3f}")


---

## 4. Log-odds (logit)

In [ ]:
log_odds = np.log(odds_default)
print(f"log-odds (logit): {log_odds:.4f}")


Negative logit → odds < 1 → probability < 0.5. Positive logit → probability > 0.5.

### 4b. Log-odds scale — why models like it

In [ ]:
# log-odds can be any real number; probability stays in (0, 1) after sigmoid
for p in [0.05, 0.25, 0.50, 0.75, 0.95]:
    o = p / (1 - p)
    print(f"P={p:.2f}  odds={o:6.2f}  logit={np.log(o):+.3f}")


---

## 5. Inverse logit — recover probability

Logistic regression outputs log-odds $\eta$; we convert back with the **sigmoid**:

$$p = \frac{1}{1 + e^{-\eta}}$$

In [ ]:
p_recovered = 1 / (1 + np.exp(-log_odds))
print(f"probability recovered from logit: {p_recovered:.4f}")
print(f"original P(default):           {p_default:.4f}")
assert abs(p_recovered - p_default) < 1e-9
print("✓ Inverse logit matches")


### 5b. Sigmoid curve

In [ ]:
eta = np.linspace(-4, 4, 100)
sigmoid = 1 / (1 + np.exp(-eta))
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(eta, sigmoid, color="steelblue")
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.6)
ax.axvline(0, color="gray", linestyle="--", alpha=0.6)
ax.set_xlabel("log-odds (η)")
ax.set_ylabel("probability p")
ax.set_title("Sigmoid — inverse of logit")
plt.tight_layout()
plt.show()


---

## 6. Worked examples

### 6a. P = 0.20 (lower risk)

In [ ]:
p_example = 0.20
odds_example = p_example / (1 - p_example)
logit_example = np.log(odds_example)

print(f"example P=0.20 -> odds: {odds_example:.4f}")
print(f"logit: {logit_example:.4f}")

p_back = 1 / (1 + np.exp(-logit_example))
print(f"recovered p: {p_back:.2f}")


At **20%** default probability, odds are **0.25** (1:4) and logit is negative.

### 6b. P = 0.80 (higher risk)

In [ ]:
p_high = 0.80
odds_high = p_high / (1 - p_high)
logit_high = np.log(odds_high)
print(f"P=0.80 → odds={odds_high:.2f}, logit={logit_high:+.3f}")


### 6c. Compare portfolio average to a sub-segment

In [ ]:
sub = df[df["dti"] >= df["dti"].median()]
p_sub = sub["default"].mean()
print(f"High-DTI segment P(default): {p_sub:.3f}")
print(f"Portfolio P(default):        {p_default:.3f}")


---

## 7. Conditional probability (warm-up)

In [ ]:
# Among grade D and E loans, what fraction default?
risky = df[df["grade"].isin(["D", "E"])]
if len(risky) > 0:
    p_risky = risky["default"].mean()
    print(f"P(default | grade in D,E): {p_risky:.3f} (n={len(risky)})")


---

## 8. Bridge to Lab 2

Logistic regression assumes:

$$\text{logit}(p) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots$$

Lab 1 used **one** overall rate $p$ for the whole portfolio. Lab 2 estimates **different** $p$ per loan based on `int_rate`, `dti`, etc.

### 8b. What changes in Lab 2?

In [ ]:
print("Lab 1: single portfolio P(default) for all 1000 rows")
print(f"  → {p_default:.4f}")
print("Lab 2: per-loan P(default) from int_rate, dti, loan_amnt, ...")
print("  → each row gets its own probability")


---

## 9. Try it yourself

1. Compute P(default) for **36 months** loans only.
2. Convert that probability to odds and log-odds.
3. Recover probability from your log-odds (inverse logit).

In [ ]:
# Your code here (optional)


In [ ]:
loans_36 = df[df["term"] == "36 months"]
p_36 = loans_36["default"].mean()
odds_36 = p_36 / (1 - p_36)
logit_36 = np.log(odds_36)
p_back_36 = 1 / (1 + np.exp(-logit_36))
print(f"36-month loans: n={len(loans_36)}, P(default)={p_36:.4f}")
print(f"odds={odds_36:.4f}, logit={logit_36:.4f}, recovered p={p_back_36:.4f}")


---

## 10. Final checkpoint

In [ ]:
print("Lab 1 — Probability exercises")
print(f"rows: {len(df)}")
print(f"P(default): {p_default:.4f}")
print(f"odds of default: {odds_default:.4f}")
print(f"log-odds (logit): {log_odds:.4f}")
print(f"example P=0.20 -> odds: {odds_example:.4f}")

assert len(df) == 1000
assert 0.45 < p_default < 0.52
print("\n✓ All checkpoint assertions passed")


---

## Reflection questions

1. If P(default) rises from 0.49 to 0.60, do odds increase or decrease?
2. Why does logistic regression model **log-odds** instead of probability directly?
3. Which loan statuses did we map to `default = 1` and why?
4. How does grade E's default rate compare to the portfolio average?

**Next:** [Lab 2 — Logistic regression](lab02_logistic_regression.ipynb)